# Taxol–Tubulin Pocket Preparation

**Working implementation — approved inputs: 1JFF, 9WDA and 9WDB.**

Run this notebook from top to bottom after installing the libraries below. It downloads those three structures, verifies tubulin and Taxol identity, expands deposited assembly 1, extracts each bound Taxol pocket, prepares capped molecular regions, and writes files for the next electronic-structure stage.

- **1JFF:** the sole approved bovine exception; allowed at 3.5 Å.
- **9WDA:** human α1A/β3 reference, 2.35 Å.
- **9WDB:** human α1A/β3 S239C variant, 2.39 Å. This is a sensitizing experimental comparator, not a patient-derived resistance mutation.

The normal run does **not** search or process the entire PDB. The optional discovery function is defined but not called.

**Boundary:** these are focused *molecular regions*, not selected electronic orbitals. The next module must calculate orbitals and resolve the returned active-space policy. No integrals, VQE or binding affinities are computed here.


## Libraries to install yourself

Use Python **3.11 or newer**. In your terminal, inside this project folder:

```bash
python -m pip install numpy gemmi rdkit jupyterlab ipykernel
```

On Windows, activate with `.venv\Scripts\activate` instead. Select the **Taxane project** notebook kernel. If using an existing notebook environment, run `%pip install numpy gemmi rdkit` yourself in a separate cell and restart its kernel.

| Library | Purpose |
|---|---|
| NumPy | Coordinates, transformations and distances |
| Gemmi | mmCIF parsing and coordinate export |
| RDKit | Chemical topology, hydrogen placement, force-field relaxation and SDF export |
| JupyterLab / ipykernel | Opening and running the notebook |

Test environment: Python 3.12, NumPy 2.5.4, Gemmi 0.7.5, RDKit 2026.09.1. The notebook does not install anything automatically. **PySCF and Qiskit are not required for this stage.** Internet access to RCSB is required on the first run; subsequent runs reuse cached files.


## Scientific choices and readiness

- The default region is **complete Taxol plus complete residues within 5 Å**. Only the surrounding pocket is retained, not the whole protein. A single bridging residue is added where needed to avoid overlapping peptide caps.
- Backbone cuts use **ACE/NME caps** anchored to the original neighboring backbone. Unsupported or missing chemistry is reported, not repaired with guessed atoms.
- Chemical Component Dictionary (CCD) definitions supply bond orders; Taxol stereochemistry is checked against the CCD.
- Hydrogens are added and optimized using UFF while all heavy atoms remain fixed. This is preparation, not an accurate protein energy calculation.
- The declared pH assumption is **7.4**. ASP/GLU are deprotonated; LYS/ARG protonated; CYS neutral unless overridden. Histidine HID/HIE/HIP scenarios are prepared separately. These are explicit hypotheses, not predicted pKa values or ranked protonation states. Other pH values require a new protonation method and are rejected.
- Waters/ions are listed in contacts but excluded from the **isolated cluster**. The deposited finite assembly is handled; an extended microtubule lattice is not invented from a heterodimer deposition.
- Local validation metrics are retained. Screening flags use Q-score <0.4, RSCC <0.7 or residue inclusion <0.7 when reported; these are conservative workflow checks, not universal accuracy criteria. Missing metrics remain unknown.
- `chemistry_checks_passed` means the implemented chemical checks passed. `ready_for_integrals` additionally requires no structural screening flags. A ready input is not proof of binding accuracy or complete lattice context.
- These complete-pocket clusters contain hundreds of atoms. A four-orbital budget is only a downstream target; this notebook does not assert that such a small electronic space adequately models them. Integral generation may still require substantial resources.


## Imports and configuration

The approved list is the default. No network requests occur in this cell.

In [ ]:
from pathlib import Path
from collections import defaultdict
from itertools import product
from urllib.request import urlopen, Request
from urllib.error import HTTPError, URLError
import hashlib
import json
import re
import time
import gzip
import csv
import xml.etree.ElementTree as ET
import numpy as np
import gemmi
from rdkit import Chem
from rdkit.Chem import AllChem

APPROVED_IDS = ("1JFF", "9WDA", "9WDB")
STANDARD_AA = set("ALA ARG ASN ASP CYS GLN GLU GLY HIS ILE LEU LYS MET PHE PRO SER THR TRP TYR VAL".split())

## Cached downloads and optional database discovery

Downloads are cached atomically. Human verification is performed on tubulin entities, not on the expression host. The discovery helper is optional and bounded; search hits still undergo validation.

In [ ]:
def download(url, destination, refresh=False):
    """Cache a successful download; never cache an error page as a structure."""
    destination = Path(destination)
    if destination.exists() and not refresh:
        return destination
    destination.parent.mkdir(parents=True, exist_ok=True)
    for attempt in range(3):
        try:
            request = Request(url, headers={"User-Agent": "TaxanePocketResearch/1.0"})
            with urlopen(request, timeout=60) as response:
                data = response.read()
            if not data:
                raise ValueError("Empty response: " + url)
            temporary = destination.with_suffix(destination.suffix + ".tmp")
            temporary.write_bytes(data)
            temporary.replace(destination)
            return destination
        except HTTPError as error:
            if error.code < 500 and error.code != 429:
                raise
            if attempt == 2:
                raise
        except (URLError, TimeoutError):
            if attempt == 2:
                raise
        time.sleep(2 ** attempt)


def search_human_taxane_ids(max_resolution=3.0, page_size=100, max_results=300):
    """Optional bounded RCSB search; entry hits still require entity validation."""
    def term(attribute, operator, value):
        return {"type": "terminal", "service": "text", "parameters": {
            "attribute": attribute, "operator": operator, "value": value}}
    nodes = [
        term("rcsb_entity_source_organism.ncbi_taxonomy_id", "equals", 9606),
        term("rcsb_nonpolymer_entity_container_identifiers.chem_comp_id", "exact_match", "TA1"),
        term("rcsb_entry_info.resolution_combined", "less_or_equal", max_resolution),
        term("rcsb_polymer_entity.pdbx_description", "contains_words", "tubulin"),
    ]
    ids, total = [], 0
    while len(ids) < max_results:
        payload = {"query": {"type": "group", "logical_operator": "and", "nodes": nodes},
                   "return_type": "entry", "request_options": {
                       "paginate": {"start": len(ids), "rows": min(page_size, max_results-len(ids))},
                       "results_content_type": ["experimental"]}}
        request = Request("https://search.rcsb.org/rcsbsearch/v2/query",
                          data=json.dumps(payload).encode(),
                          headers={"Content-Type": "application/json"})
        with urlopen(request, timeout=60) as response:
            if response.status == 204:
                break
            result = json.load(response)
        total = result.get("total_count", 0)
        page = [row["identifier"] for row in result.get("result_set", [])]
        ids.extend(page)
        if not page or len(ids) >= total:
            break
    return {"ids": ids, "total_matches": total, "truncated": len(ids) < total}


def rows(block, category):
    """Convert a CIF category to ordinary row dictionaries."""
    columns = block.get_mmcif_category(category)
    if not columns:
        return []
    return [dict(zip(columns, values)) for values in zip(*columns.values())]


def number(value, default=None):
    if value is None or isinstance(value, bool):
        return default
    try:
        return float(value)
    except (ValueError, TypeError):
        return default


def inspect_entry(block, pdb_id, max_resolution):
    """Verify source species on tubulin entities, never on the expression host."""
    entities = {x["id"]: x.get("pdbx_description", "") for x in rows(block, "_entity.")}
    species = defaultdict(set)
    for category, field in [("_entity_src_gen.", "pdbx_gene_src_ncbi_taxonomy_id"),
                            ("_entity_src_nat.", "pdbx_ncbi_taxonomy_id")]:
        for row in rows(block, category):
            if row.get(field):
                species[row["entity_id"]].add(str(row[field]))
    beta = [eid for eid, name in entities.items()
            if "tubulin" in name.lower() and "beta" in name.lower()]
    tubulin = [eid for eid, name in entities.items() if "tubulin" in name.lower()]
    if not beta:
        raise ValueError("No identifiable beta-tubulin entity.")
    allowed_species = "9913" if pdb_id == "1JFF" else "9606"
    if any(species[eid] != {allowed_species} for eid in tubulin):
        raise ValueError("Tubulin source is not human (only bovine 1JFF is allowed).")
    methods = [x.get("method", "") for x in rows(block, "_exptl.")]
    resolutions = [number(x.get("resolution")) for x in rows(block, "_em_3d_reconstruction.")]
    resolutions += [number(x.get("ls_d_res_high")) for x in rows(block, "_refine.")]
    resolutions = [x for x in resolutions if x is not None and x > 0]
    if not methods or not resolutions:
        raise ValueError("Experimental method or resolution is missing.")
    resolution = min(resolutions)
    # Explicit legacy exception: 1JFF is 3.5 Å, above the human default of 3.0 Å.
    limit = 3.5 if pdb_id == "1JFF" else max_resolution
    if resolution > limit:
        raise ValueError(f"Resolution {resolution} exceeds allowed {limit} Å.")
    if not any(x.get("comp_id") == "TA1" for x in rows(block, "_pdbx_entity_nonpoly.")):
        raise ValueError("The entry does not contain paclitaxel component TA1.")
    return {"beta_entities": beta, "entities": entities,
            "species": {k: sorted(v) for k, v in species.items()},
            "resolution_angstrom": resolution, "methods": methods,
            "nonhuman_exception": pdb_id == "1JFF",
            "mutations": rows(block, "_entity."),
            "sequence_references": rows(block, "_struct_ref.")}

## Assembly expansion and local contact extraction

Apply deposited operators and enumerate every bound TA1 instance in the requested finite assemblies/models. Keep source identifiers and coherent alternate conformations. Unoccupied sites and docking are outside this bound-complex workflow.

In [ ]:
def operation_sequences(expression):
    """Expand deposited operation ranges and products, such as (1-3)(4,5)."""
    groups = re.findall(r"\(([^()]+)\)", expression) or [expression]
    expanded = []
    for group in groups:
        values = []
        for item in group.split(","):
            item = item.strip()
            if re.fullmatch(r"\d+-\d+", item):
                first, last = map(int, item.split("-"))
                values.extend(str(x) for x in range(first, last + 1))
            else:
                values.append(item)
        expanded.append(values)
    return list(product(*expanded))


def assembly_atoms(block, assembly_id="1", model_id="1", max_atoms=300000):
    """Apply CIF assembly operators and keep original atom identifiers."""
    source = []
    for row in rows(block, "_atom_site."):
        if str(row.get("pdbx_PDB_model_num", "1")) != str(model_id):
            continue
        if row.get("type_symbol", "").upper() in {"H", "D"}:
            continue  # Hydrogen placement is regenerated during preparation.
        seq = row.get("label_seq_id") or row.get("auth_seq_id")
        source.append({"chain": row["label_asym_id"], "auth_chain": row.get("auth_asym_id"),
                       "entity": row["label_entity_id"], "seq": str(seq),
                       "auth_seq": row.get("auth_seq_id"), "insertion": row.get("pdbx_PDB_ins_code"),
                       "resname": row["label_comp_id"], "name": row["label_atom_id"],
                       "element": row["type_symbol"], "alt": row.get("label_alt_id") or "",
                       "occupancy": number(row.get("occupancy"), 0),
                       "b_factor": number(row.get("B_iso_or_equiv")),
                       "xyz": np.array([float(row[k]) for k in ("Cartn_x", "Cartn_y", "Cartn_z")]),
                       "source_id": row["id"]})
    # Resolve each residue coherently; blank atoms coexist with one selected alt.
    grouped = defaultdict(list)
    for atom in source:
        grouped[(atom["chain"], atom["seq"], atom["insertion"], atom["resname"])].append(atom)
    selected = []
    for atoms in grouped.values():
        scores = defaultdict(float)
        for atom in atoms:
            if atom["alt"]:
                scores[atom["alt"]] += atom["occupancy"]
        choice = sorted(scores, key=lambda x: (-scores[x], x))[0] if scores else ""
        for atom in atoms:
            if atom["alt"] in {"", choice}:
                atom["alternate_present"] = bool(scores)
                selected.append(atom)
    operations = {}
    for row in rows(block, "_pdbx_struct_oper_list."):
        matrix = np.eye(4)
        for i in range(3):
            matrix[i, 3] = float(row[f"vector[{i+1}]"])
            for j in range(3):
                matrix[i, j] = float(row[f"matrix[{i+1}][{j+1}]"])
        operations[row["id"]] = matrix
    generators = [x for x in rows(block, "_pdbx_struct_assembly_gen.")
                  if str(x["assembly_id"]) == str(assembly_id)]
    if not generators:
        raise ValueError(f"Deposited assembly {assembly_id} is unavailable.")
    output, seen = [], set()
    for generator in generators:
        chains = set(generator["asym_id_list"].split(","))
        for sequence in operation_sequences(generator["oper_expression"]):
            transform = np.eye(4)
            for operation in sequence:
                transform = transform @ operations[operation]
            for atom in selected:
                if atom["chain"] not in chains:
                    continue
                xyz = (transform @ np.append(atom["xyz"], 1))[:3]
                key = (atom["source_id"], tuple(np.round(xyz, 5)))
                if key in seen:
                    continue
                seen.add(key)
                copy = dict(atom, xyz=xyz, operation="*".join(sequence))
                copy["residue_key"] = f'{copy["chain"]}:{copy["operation"]}:{copy["seq"]}:{copy["insertion"]}:{copy["resname"]}'
                output.append(copy)
                if len(output) > max_atoms:
                    raise ValueError("Assembly exceeds configured atom budget; choose a smaller deposited assembly.")
    return output


def extract_pockets(atoms, beta_entities, radius):
    """Enumerate bound TA1 sites and select only complete local residues."""
    residues = defaultdict(list)
    for atom in atoms:
        residues[atom["residue_key"]].append(atom)
    pockets = []
    for ligand_key, ligand in residues.items():
        if ligand[0]["resname"] != "TA1":
            continue
        ligand_xyz = np.array([a["xyz"] for a in ligand])
        contacts, selected, nearest_beta = [], [], []
        for key, residue in residues.items():
            if key == ligand_key:
                continue
            xyz = np.array([a["xyz"] for a in residue])
            distances = np.linalg.norm(xyz[:, None, :] - ligand_xyz[None, :, :], axis=2)
            i, j = np.unravel_index(distances.argmin(), distances.shape)
            distance = float(distances[i, j])
            if residue[0]["entity"] in beta_entities:
                nearest_beta.append((distance, residue[0]["chain"], residue[0]["operation"]))
            if distance <= radius:
                contacts.append({"residue_key": key, "resname": residue[0]["resname"],
                                 "protein_atom": residue[i]["name"], "ligand_atom": ligand[j]["name"],
                                 "distance_angstrom": distance,
                                 "within_4A": distance <= 4, "within_5A": distance <= 5})
                if residue[0]["resname"] in STANDARD_AA:
                    selected.append(key)
        if not nearest_beta or min(nearest_beta)[0] > radius:
            pockets.append({"ligand_key": ligand_key, "status": "rejected",
                            "reasons": ["TA1 is not near a verified beta-tubulin entity."]})
            continue
        pockets.append({"ligand_key": ligand_key, "selected_residues": selected,
                        "contacts": contacts, "beta_instance": min(nearest_beta)[1:],
                        "status": "extracted", "reasons": [], "ligand": ligand,
                        "residue_lookup": residues})
    # Include a single intervening residue to prevent overlapping cap atoms.
    for pocket in pockets:
        if pocket["status"] != "extracted":
            continue
        selected = set(pocket["selected_residues"])
        by_position = {(r[0]["chain"], r[0]["operation"], int(r[0]["seq"])): k
                       for k, r in residues.items() if r[0]["resname"] in STANDARD_AA}
        additions = []
        for (chain, operation, seq), key in by_position.items():
            if key not in selected and by_position.get((chain, operation, seq-1)) in selected and by_position.get((chain, operation, seq+1)) in selected:
                additions.append(key)
        pocket["bridging_residues"] = additions
        pocket["selected_residues"] += additions
    return pockets

## CCD chemistry and controlled protonation

Use chemical definitions rather than inferring all bonds from distances. Histidine scenarios are enumerated by the public function; explicit overrides can select a scenario.

In [ ]:
def component_definition(component, cache_dir):
    """Use deposited chemical topology, not guessed distance-based bonds."""
    path = download(f"https://files.rcsb.org/ligands/download/{component}.cif",
                    Path(cache_dir) / "ccd" / f"{component}.cif")
    block = gemmi.cif.read_file(str(path)).sole_block()
    return rows(block, "_chem_comp_atom."), rows(block, "_chem_comp_bond.")


def add_component(builder, coordinates, component, cache_dir, mapping, positions):
    """Add all observed heavy atoms with CCD bonds and formal charges."""
    definitions, bonds = component_definition(component, cache_dir)
    expected = {a["atom_id"]: a for a in definitions
                if a["type_symbol"] not in {"H", "D"} and not (component in STANDARD_AA and a["atom_id"] == "OXT")}
    actual = {a["name"]: a for a in coordinates}
    if len(actual) != len(coordinates):
        raise ValueError(f"Duplicate atom names in {component}.")
    if set(actual) != set(expected):
        raise ValueError(f"{component}: missing {sorted(set(expected)-set(actual))}, unexpected {sorted(set(actual)-set(expected))}")
    indices = {}
    for name, definition in expected.items():
        atom = Chem.Atom(definition["type_symbol"])
        atom.SetFormalCharge(int(number(definition.get("charge"), 0)))
        # Backbone nitrogens become neutral peptide amides after cap/link assembly.
        if component in STANDARD_AA and name == "N":
            atom.SetFormalCharge(0)
        index = builder.AddAtom(atom)
        indices[name] = index
        record = actual[name]
        positions.append(record["xyz"])
        mapping.append({"kind": "experimental", "source_id": record["source_id"],
                        "residue_key": record["residue_key"], "atom_name": name,
                        "operation": record["operation"],
                        "expected_stereo": definition.get("pdbx_stereo_config") if component == "TA1" else None})
    orders = {"SING": Chem.BondType.SINGLE, "DOUB": Chem.BondType.DOUBLE,
              "TRIP": Chem.BondType.TRIPLE, "AROM": Chem.BondType.AROMATIC}
    for bond in bonds:
        left, right = bond["atom_id_1"], bond["atom_id_2"]
        if left in indices and right in indices:
            builder.AddBond(indices[left], indices[right], orders[bond["value_order"]])
    return indices


def apply_protonation(builder, indices, residue_name, choice=None):
    """Explicit conventional pH-7.4 microstates; HIS/CYS require a choice."""
    defaults = {"ASP": "deprotonated", "GLU": "deprotonated", "LYS": "protonated", "ARG": "protonated"}
    if residue_name in {"HIS", "CYS"} and choice is None:
        raise ValueError(f"{residue_name} protonation is ambiguous; supply a residue-key override.")
    choice = choice or defaults.get(residue_name, "neutral")
    if residue_name in {"ASP", "GLU"}:
        center, oxygen = ("CG", "OD2") if residue_name == "ASP" else ("CD", "OE2")
        first = "OD1" if residue_name == "ASP" else "OE1"
        if choice not in {"deprotonated", "neutral"}:
            raise ValueError("Acid choice must be deprotonated or neutral.")
        builder.GetBondBetweenAtoms(indices[center], indices[first]).SetBondType(Chem.BondType.DOUBLE)
        builder.GetBondBetweenAtoms(indices[center], indices[oxygen]).SetBondType(Chem.BondType.SINGLE)
        builder.GetAtomWithIdx(indices[first]).SetFormalCharge(0)
        builder.GetAtomWithIdx(indices[oxygen]).SetFormalCharge(-1 if choice == "deprotonated" else 0)
    elif residue_name == "LYS":
        if choice not in {"protonated", "neutral"}:
            raise ValueError("LYS choice must be protonated or neutral.")
        builder.GetAtomWithIdx(indices["NZ"]).SetFormalCharge(1 if choice == "protonated" else 0)
    elif residue_name == "ARG":
        if choice != "protonated":
            raise ValueError("Only protonated ARG is supported by this template.")
        # Guanidinium charge is assigned to its double-bonded terminal nitrogen.
        for name in ("NE", "NH1", "NH2"):
            builder.GetAtomWithIdx(indices[name]).SetFormalCharge(0)
        double_n = next(n for n in ("NH1", "NH2") if builder.GetBondBetweenAtoms(indices["CZ"], indices[n]).GetBondTypeAsDouble() == 2)
        builder.GetAtomWithIdx(indices[double_n]).SetFormalCharge(1)
    elif residue_name == "HIS":
        if choice not in {"HID", "HIE", "HIP"}:
            raise ValueError("HIS choices: HID (ND1-H), HIE (NE2-H), HIP (both).")
        ring = [("CG", "ND1"), ("ND1", "CE1"), ("CE1", "NE2"), ("NE2", "CD2"), ("CD2", "CG")]
        for left, right in ring:
            bond = builder.GetBondBetweenAtoms(indices[left], indices[right])
            bond.SetBondType(Chem.BondType.AROMATIC)
            bond.SetIsAromatic(True)
        for name in ("CG", "ND1", "CE1", "NE2", "CD2"):
            builder.GetAtomWithIdx(indices[name]).SetIsAromatic(True)
        for name in ("ND1", "NE2"):
            atom = builder.GetAtomWithIdx(indices[name])
            has_h = choice == "HIP" or (name == "ND1" and choice == "HID") or (name == "NE2" and choice == "HIE")
            atom.SetFormalCharge(1 if choice == "HIP" and name == "NE2" else 0)
            atom.SetNumExplicitHs(int(has_h))
            atom.SetNoImplicit(True)
    elif residue_name == "CYS":
        if choice not in {"neutral", "deprotonated"}:
            raise ValueError("CYS choice must be neutral or deprotonated.")
        builder.GetAtomWithIdx(indices["SG"]).SetFormalCharge(-1 if choice == "deprotonated" else 0)
    elif choice != "neutral":
        raise ValueError(f"Unsupported microstate for {residue_name}: {choice}")
    return choice

## Peptide capping, hydrogen optimization and validation

Preserve complete functional groups, link selected peptide segments and cap their exposed ends. Save selected coordinates even if a later check blocks readiness.

In [ ]:
def prepare_region(pocket, cache_dir, protonation_overrides, max_atoms=1800):
    """Prepare full TA1 plus selected peptide segments, with ACE/NME end caps."""
    lookup, selected = pocket["residue_lookup"], pocket["selected_residues"]
    builder, positions, mapping, indices = Chem.RWMol(), [], [], {}
    caps, microstates = [], {}
    add_component(builder, pocket["ligand"], "TA1", cache_dir, mapping, positions)
    for key in selected:
        atoms = lookup[key]
        indices[key] = add_component(builder, atoms, atoms[0]["resname"], cache_dir, mapping, positions)
        microstates[key] = apply_protonation(builder, indices[key], atoms[0]["resname"], protonation_overrides.get(key))
    # Original backbone atoms determine peptide neighbors without renumbering.
    residue_by_seq = {(r[0]["chain"], r[0]["operation"], r[0]["seq"]): key
                      for key, r in lookup.items() if r[0]["resname"] in STANDARD_AA}
    def neighbor(key, offset):
        atom = lookup[key][0]
        seq = int(atom["seq"]) + offset
        return residue_by_seq.get((atom["chain"], atom["operation"], str(seq)))
    def atom_record(key, name):
        if key is None:
            raise ValueError("Missing backbone neighbor: cannot construct a geometry-based cap.")
        candidates = [a for a in lookup[key] if a["name"] == name]
        if len(candidates) != 1:
            raise ValueError(f"Missing/ambiguous cap anchor {key}:{name}.")
        return candidates[0]
    def link(left_key, right_key):
        a, b = atom_record(left_key, "C"), atom_record(right_key, "N")
        if not 1.1 < np.linalg.norm(a["xyz"] - b["xyz"]) < 1.7:
            raise ValueError("Broken peptide geometry at a truncation boundary.")
    def cap_atom(key, name, element, cap_kind):
        record = atom_record(key, name)
        index = builder.AddAtom(Chem.Atom(element))
        positions.append(record["xyz"])
        mapping.append({"kind": "cap", "cap": cap_kind, "source_anchor_id": record["source_id"],
                        "residue_key": key, "atom_name": name})
        return index
    for key in selected:
        previous, following = neighbor(key, -1), neighbor(key, 1)
        link(previous, key)
        link(key, following)
        if previous in indices:
            builder.AddBond(indices[previous]["C"], indices[key]["N"], Chem.BondType.SINGLE)
        else:
            # ACE replaces the excluded previous residue: methyl-C(=O)-N.
            methyl = cap_atom(previous, "CA", "C", "ACE")
            carbon = cap_atom(previous, "C", "C", "ACE")
            oxygen = cap_atom(previous, "O", "O", "ACE")
            builder.AddBond(methyl, carbon, Chem.BondType.SINGLE)
            builder.AddBond(carbon, oxygen, Chem.BondType.DOUBLE)
            builder.AddBond(carbon, indices[key]["N"], Chem.BondType.SINGLE)
            caps.append({"type": "ACE", "attached_to": key})
        if following not in indices:
            # NME replaces the excluded next residue: C(=O)-NH-methyl.
            nitrogen = cap_atom(following, "N", "N", "NME")
            methyl = cap_atom(following, "CA", "C", "NME")
            builder.AddBond(indices[key]["C"], nitrogen, Chem.BondType.SINGLE)
            builder.AddBond(nitrogen, methyl, Chem.BondType.SINGLE)
            caps.append({"type": "NME", "attached_to": key})
    molecule = builder.GetMol()
    Chem.SanitizeMol(molecule)
    conformer = Chem.Conformer(len(positions))
    for i, xyz in enumerate(positions):
        conformer.SetAtomPosition(i, tuple(map(float, xyz)))
    molecule.AddConformer(conformer)
    Chem.AssignStereochemistryFrom3D(molecule)
    Chem.AssignStereochemistry(molecule, cleanIt=True, force=True)
    for index, provenance in enumerate(mapping):
        expected = provenance.get("expected_stereo")
        if expected in {"R", "S"}:
            atom = molecule.GetAtomWithIdx(index)
            if not atom.HasProp("_CIPCode") or atom.GetProp("_CIPCode") != expected:
                raise ValueError("Taxol stereochemistry disagrees with the CCD at " + provenance["atom_name"])
    # Preserve bound heavy atoms, add hydrogens, and relax only the new hydrogens.
    heavy_count = molecule.GetNumAtoms()
    molecule = Chem.AddHs(molecule, addCoords=True)
    if molecule.GetNumAtoms() > max_atoms:
        raise ValueError("Prepared region exceeds configured atom budget.")
    if not AllChem.UFFHasAllMoleculeParams(molecule):
        raise ValueError("UFF lacks parameters; hydrogens cannot be optimized safely.")
    force_field = AllChem.UFFGetMoleculeForceField(molecule, ignoreInterfragInteractions=False)
    for i in range(heavy_count):
        force_field.AddFixedPoint(i)
    force_field.Initialize()
    converged = force_field.Minimize(maxIts=2000) == 0
    for i in range(heavy_count, molecule.GetNumAtoms()):
        parent = molecule.GetAtomWithIdx(i).GetNeighbors()[0].GetIdx()
        mapping.append({"kind": "added_hydrogen", "parent_index": parent})
    charge = sum(a.GetFormalCharge() for a in molecule.GetAtoms())
    electrons = sum(a.GetAtomicNum() for a in molecule.GetAtoms()) - charge
    radicals = sum(a.GetNumRadicalElectrons() for a in molecule.GetAtoms())
    if radicals or electrons % 2:
        raise ValueError("This closed-shell preparation produced radicals or an odd electron count.")
    xyz = molecule.GetConformer().GetPositions()
    if not np.isfinite(xyz).all():
        raise ValueError("Invalid coordinates after hydrogen preparation.")
    # A severe nonbonded clash blocks readiness; bonded/1-3 pairs are excluded.
    topological = Chem.GetDistanceMatrix(molecule)
    distances = np.linalg.norm(xyz[:, None] - xyz[None, :], axis=2)
    bad = np.argwhere(np.triu((distances < 1.0) & (topological > 2), 1))
    warnings = []
    periodic_table = Chem.GetPeriodicTable()
    for bond in molecule.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        ideal = periodic_table.GetRcovalent(molecule.GetAtomWithIdx(i).GetAtomicNum()) + periodic_table.GetRcovalent(molecule.GetAtomWithIdx(j).GetAtomicNum())
        if not 0.65 * ideal < distances[i, j] < 1.35 * ideal:
            warnings.append(f"Unusual bond length for atoms {i}-{j}.")
    if not converged:
        warnings.append("Hydrogen relaxation did not converge.")
    if len(bad):
        warnings.append(f"{len(bad)} severe nonbonded atom clashes (<1 Å).")
    elements = [a.GetSymbol() for a in molecule.GetAtoms()]
    region = {"elements": elements, "coordinates_angstrom": xyz.tolist(),
              "bonds": [(b.GetBeginAtomIdx(), b.GetEndAtomIdx(), b.GetBondTypeAsDouble()) for b in molecule.GetBonds()],
              "formal_charges": [a.GetFormalCharge() for a in molecule.GetAtoms()],
              "net_charge": charge, "spin_2s": 0, "electron_count": electrons,
              "source_atom_mapping": mapping, "caps": caps, "protonation": microstates,
              "environment": "isolated_cluster", "ready_for_integrals": not warnings,
              "chemistry_checks_passed": not warnings,
              "warnings": warnings, "orbital_indices": None, "active_electrons": None,
              "molecular_region": "complete_taxol_plus_capped_local_peptide_segments"}
    return molecule, region

def save_pocket_files(atoms, contacts, folder):
    """Export inspectable coordinates and a compact contact table."""
    document = gemmi.cif.Document()
    block = document.add_new_block("selected_pocket")
    tags = ["group_PDB", "id", "type_symbol", "label_atom_id", "label_alt_id",
            "label_comp_id", "label_asym_id", "label_entity_id", "label_seq_id",
            "Cartn_x", "Cartn_y", "Cartn_z", "occupancy", "B_iso_or_equiv",
            "auth_asym_id", "auth_seq_id", "pdbx_PDB_model_num"]
    loop = block.init_loop("_atom_site.", tags)
    for i, atom in enumerate(atoms, 1):
        chain = atom["chain"] + "_" + atom["operation"]
        values = ["ATOM" if atom["resname"] in STANDARD_AA else "HETATM", str(i),
                  atom["element"], atom["name"], ".", atom["resname"], chain,
                  atom["entity"], atom["seq"] if atom["resname"] in STANDARD_AA else ".",
                  *[f"{x:.6f}" for x in atom["xyz"]], str(atom["occupancy"]),
                  str(atom["b_factor"] or 0), chain, str(atom["auth_seq"]), "1"]
        loop.add_row([value if value == "." else gemmi.cif.quote(value) for value in values])
    document.write_file(str(folder / "pocket.cif"))
    with (folder / "contacts.csv").open("w", newline="") as stream:
        fields = ["residue_key", "resname", "protein_atom", "ligand_atom", "distance_angstrom", "within_4A", "within_5A"]
        writer = csv.DictWriter(stream, fieldnames=fields)
        writer.writeheader()
        writer.writerows(contacts)

## Public integration function and serialization

This is the public entry point. It returns ordinary dictionaries and arrays/lists, making integration with another notebook or module straightforward.

In [ ]:
def json_safe(value):
    """Convert NumPy values and paths before serializing provenance."""
    if isinstance(value, np.ndarray):
        return value.tolist()
    if isinstance(value, np.generic):
        return value.item()
    if isinstance(value, Path):
        return str(value)
    raise TypeError(type(value).__name__)


def prepare_human_taxane_regions(*, pdb_ids=APPROVED_IDS, max_resolution_angstrom=3.0,
                                 contact_radius_angstrom=5.0, assembly_ids=("1",),
                                 protonation_overrides=None, ph=7.4,
                                 max_spatial_orbitals=4, cache_dir="data/structures",
                                 output_dir="results", max_region_atoms=1800):
    """Download, verify and prepare approved taxane pockets; no integral calculation.

    pdb_ids=None explicitly enables bounded database discovery. The default run
    accesses only 1JFF, 9WDA and 9WDB. 1JFF alone has species/resolution exceptions.
    """
    if not 0 < contact_radius_angstrom <= 8:
        raise ValueError("Contact radius must be in (0, 8] Å.")
    if not np.isfinite(max_resolution_angstrom) or max_resolution_angstrom <= 0:
        raise ValueError("Resolution cutoff must be finite and positive.")
    if ph != 7.4:
        raise ValueError("Automatic templates support the declared pH 7.4 assumption only.")
    if not isinstance(max_spatial_orbitals, int) or max_spatial_orbitals < 1:
        raise ValueError("Orbital budget must be a positive integer.")
    cache_dir, output_dir = Path(cache_dir), Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    overrides = protonation_overrides or {}
    discovery = search_human_taxane_ids(max_resolution_angstrom) if pdb_ids is None else None
    identifiers = discovery["ids"] if discovery is not None else list(pdb_ids)
    result = {"schema_version": "1.0", "discovery": discovery,
              "configuration": {"pdb_ids": identifiers, "contact_radius_angstrom": contact_radius_angstrom,
                                "max_resolution_angstrom": max_resolution_angstrom,
                                "assembly_ids": assembly_ids, "ph_assumption": ph,
                                "protonation_overrides": overrides},
              "accepted_entries": [], "excluded_entries": [], "retrieval_errors": [], "pockets": []}
    for raw_id in dict.fromkeys(identifiers):
        pdb_id = raw_id.upper()
        if not re.fullmatch(r"[0-9A-Z]{4}", pdb_id):
            raise ValueError("Invalid PDB identifier: " + pdb_id)
        try:
            path = download(f"https://files.rcsb.org/download/{pdb_id}.cif", cache_dir / f"{pdb_id}.cif")
            block = gemmi.cif.read_file(str(path)).sole_block()
        except Exception as error:
            result["retrieval_errors"].append({"pdb_id": pdb_id, "error": str(error)})
            continue
        try:
            metadata = inspect_entry(block, pdb_id, max_resolution_angstrom)
        except ValueError as error:
            result["excluded_entries"].append({"pdb_id": pdb_id, "reason": str(error)})
            continue
        metadata.update(pdb_id=pdb_id, sha256=hashlib.sha256(path.read_bytes()).hexdigest(),
                        source_url=f"https://files.rcsb.org/download/{pdb_id}.cif")
        validation_rows = []
        # Validation report availability is separate from geometric checks.
        lower = pdb_id.lower()
        try:
            report = download(f"https://files.rcsb.org/pub/pdb/validation_reports/{lower[1:3]}/{lower}/{lower}_validation.xml.gz",
                              cache_dir / f"{pdb_id}_validation.xml.gz")
            validation_root = ET.fromstring(gzip.decompress(report.read_bytes()))
            validation_rows = [dict(x.attrib) for x in validation_root.iter("ModelledSubgroup")]
            metadata["validation_report"] = str(report)
        except Exception as error:
            metadata["validation_report_warning"] = str(error)
        result["accepted_entries"].append(metadata)
        model_ids = sorted({str(r.get("pdbx_PDB_model_num", "1")) for r in rows(block, "_atom_site.")})
        for model_id in model_ids:
            for assembly_id in assembly_ids:
                try:
                    atoms = assembly_atoms(block, assembly_id, model_id)
                    pockets = extract_pockets(atoms, metadata["beta_entities"], contact_radius_angstrom)
                except Exception as error:
                    result["pockets"].append({"pdb_id": pdb_id, "assembly_id": assembly_id,
                                             "model_id": model_id, "status": "assembly_failed", "reasons": [str(error)]})
                    continue
                for index, pocket in enumerate(pockets):
                    pocket_id = f"{pdb_id}_assembly{assembly_id}_model{model_id}_pocket{index+1}"
                    record = {k: v for k, v in pocket.items() if k not in {"ligand", "residue_lookup"}}
                    record.update(pocket_id=pocket_id, pdb_id=pdb_id, assembly_id=assembly_id, model_id=model_id, regions=[])
                    result["pockets"].append(record)
                    if pocket["status"] != "extracted":
                        continue
                    folder = output_dir / pocket_id
                    folder.mkdir(parents=True, exist_ok=True)
                    chosen_atoms = pocket["ligand"] + [a for k in pocket["selected_residues"] for a in pocket["residue_lookup"][k]]
                    # Save exact selected atoms even if later chemistry cannot be resolved.
                    (folder / "selected_atoms.json").write_text(json.dumps(chosen_atoms, default=json_safe, indent=2))
                    save_pocket_files(chosen_atoms, pocket["contacts"], folder)
                    record["files"] = {"pocket_cif": str(folder / "pocket.cif"), "contacts_csv": str(folder / "contacts.csv")}
                    record["excluded_nearby_components"] = [c for c in pocket["contacts"] if c["resname"] not in STANDARD_AA]
                    structural_warnings = []
                    unknown = [c["resname"] for c in pocket["contacts"]
                               if c["resname"] not in STANDARD_AA | {"HOH", "MG", "ZN", "NA", "CL", "CA", "GTP", "GDP", "G2P"}]
                    if unknown:
                        structural_warnings.append("Unmodeled nearby chemical components: " + ", ".join(sorted(set(unknown))))
                    if any(a["occupancy"] < 0.9 for a in chosen_atoms):
                        structural_warnings.append("Selected atoms have occupancy below 0.9.")
                    if any(a["alternate_present"] for a in chosen_atoms):
                        structural_warnings.append("Alternate conformations require inspection.")
                    # Finite deposited assemblies do not prove a complete helical neighborhood.
                    if any("MICROSCOPY" in method for method in metadata["methods"]):
                        record["context_note"] = "Deposited finite assembly only; extended microtubule lattice is not reconstructed."
                    selected_keys = {(a["auth_chain"], str(a["auth_seq"]), a["resname"]) for a in chosen_atoms}
                    local_validation = [row for row in validation_rows
                                        if (row.get("chain"), row.get("resnum"), row.get("resname")) in selected_keys
                                        and row.get("model") == model_id]
                    record["local_validation"] = local_validation
                    record["local_density_quality"] = "reported_metrics_only" if local_validation else "unknown"
                    for row in local_validation:
                        qscore = number(row.get("Q_score"))
                        rscc = number(row.get("rscc"))
                        inclusion = number(row.get("residue_inclusion"))
                        if (qscore is not None and qscore < 0.4) or (rscc is not None and rscc < 0.7) or (inclusion is not None and inclusion < 0.7):
                            structural_warnings.append(f"Local density screening flag: {row.get('chain')}:{row.get('resnum')} {row.get('resname')}")
                    record["structural_warnings"] = structural_warnings
                    try:
                        local_overrides = overrides.get(pdb_id, overrides)
                        unresolved = [key for key in pocket["selected_residues"]
                                      if pocket["residue_lookup"][key][0]["resname"] == "HIS"
                                      and key not in local_overrides]
                        if len(unresolved) > 2:
                            raise ValueError("More than nine histidine scenarios: provide explicit overrides.")
                        # Enumerate plausible histidine states rather than silently selecting one.
                        scenarios = list(product(("HID", "HIE", "HIP"), repeat=len(unresolved)))
                        scenario_overrides = dict(local_overrides)
                        for key in pocket["selected_residues"]:
                            if pocket["residue_lookup"][key][0]["resname"] == "CYS":
                                scenario_overrides.setdefault(key, "neutral")
                        record["protonation_scenarios"] = len(scenarios)
                        for scenario_number, choices in enumerate(scenarios, 1):
                            chosen_overrides = dict(scenario_overrides, **dict(zip(unresolved, choices)))
                            try:
                                molecule, region = prepare_region(pocket, cache_dir, chosen_overrides, max_atoms=max_region_atoms)
                            except Exception as error:
                                record["reasons"].append(f"Scenario {scenario_number}: {error}")
                                continue
                            region["protonation_scenario"] = dict(zip(unresolved, choices))
                            region["protonation_note"] = "Explicit microstate hypothesis, not a pKa prediction. CYS neutral unless overridden."
                            region["region_id"] = pocket_id + f"_region{scenario_number}"
                            region["orbital_selection_policy"] = {"max_spatial_orbitals": max_spatial_orbitals,
                                                                   "method": "downstream_contact_orbital_analysis"}
                            region["warnings"].extend(structural_warnings)
                            region["ready_for_integrals"] = not region["warnings"]
                            region["readiness_scope"] = "isolated_prepared_cluster; not a validated complete lattice model"
                            sdf = folder / f"region_{scenario_number}.sdf"
                            with Chem.SDWriter(str(sdf)) as writer:
                                writer.write(molecule)
                            xyz_file = folder / f"region_{scenario_number}.xyz"
                            Chem.MolToXYZFile(molecule, str(xyz_file))
                            region["files"] = {"sdf": str(sdf), "xyz": str(xyz_file)}
                            record["regions"].append(region)
                        record["status"] = "ready" if any(r["ready_for_integrals"] for r in record["regions"]) else "needs_review"
                    except Exception as error:
                        record["status"] = "needs_review"
                        record["reasons"].append(str(error))
                    (folder / "pocket.json").write_text(json.dumps(record, default=json_safe, indent=2))
        print(f"Processed {pdb_id}: {metadata['resolution_angstrom']} Å")
    result["coverage"] = {"accepted_entries": len(result["accepted_entries"]),
                          "excluded_entries": len(result["excluded_entries"]),
                          "retrieval_errors": len(result["retrieval_errors"]),
                          "pocket_records": len(result["pockets"]),
                          "prepared_scenarios": sum(len(p.get("regions", [])) for p in result["pockets"]),
                          "needs_review_pockets": sum(p["status"] == "needs_review" for p in result["pockets"]),
                          "ready_regions": sum(r["ready_for_integrals"] for p in result["pockets"] for r in p.get("regions", []))}
    (output_dir / "manifest.json").write_text(json.dumps(result, default=json_safe, indent=2))
    return result


def as_pyscf_kwargs(region, basis="def2-svp", *, allow_structural_warnings=False):
    """Allow reviewed-risk handoff explicitly; never bypass chemistry failures."""
    if not region.get("chemistry_checks_passed", False):
        raise ValueError("Region failed chemistry checks and cannot be handed off.")
    if not region["ready_for_integrals"] and not allow_structural_warnings:
        raise ValueError("Region requires review before electronic-structure calculations.")
    if region["environment"] != "isolated_cluster":
        raise ValueError("An embedded model requires an explicit embedding adapter.")
    return {"atom": list(zip(region["elements"], region["coordinates_angstrom"])),
            "unit": "Angstrom", "charge": region["net_charge"],
            "spin": region["spin_2s"], "basis": basis}

## Run only the approved structures

This cell performs real downloads and writes `data/structures/` and `results/` relative to the notebook kernel's current working directory. Check the printed path before running. The first run may take several minutes; cached reruns avoid repeated downloads. Existing result files for the same run are updated.

The default uses assembly 1 and all deposited models. Each alternate histidine microstate is a separate region. You can reduce `contact_radius_angstrom` after inspecting the contact lists, but doing so changes the scientific model.


In [ ]:
# Confirm where files will be saved; change PROJECT_DIR if needed.
PROJECT_DIR = Path.cwd()
print("Project directory:", PROJECT_DIR)

# Leave this empty to enumerate histidine states automatically.
# Example override format: {"9WDA": {"B:1:227:None:HIS": "HIE"}}.
PROTONATION_OVERRIDES = {}

# Run the three approved IDs only; None would explicitly enable discovery.
result = prepare_human_taxane_regions(
    pdb_ids=APPROVED_IDS,
    max_resolution_angstrom=3.0,
    contact_radius_angstrom=5.0,
    assembly_ids=("1",),
    protonation_overrides=PROTONATION_OVERRIDES,
    max_spatial_orbitals=4,
    cache_dir=PROJECT_DIR / "data" / "structures",
    output_dir=PROJECT_DIR / "results",
)
print(json.dumps(result["coverage"], indent=2))

## Inspect results before handoff

Each pocket directory contains `pocket.cif`, `contacts.csv`, `selected_atoms.json`, `pocket.json`, and SDF/XYZ files for successfully built scenarios. `results/manifest.json` contains the full return object and rejection reasons.

**Observed validation run:** three accepted structures, three bound pockets and nine prepared scenarios. The 1JFF scenarios passed the implemented checks. The human scenarios were chemically prepared but flagged for local density review at ARG 276 (both structures) and ARG 359 (9WDA). This is a meaningful result; do not clear these flags merely to obtain a ready output. Inspect the validation report/maps and revise the scientific region or structural preparation if appropriate.


In [ ]:
# Print compact, actionable diagnostics for every extracted site.
for pocket in result["pockets"]:
    print("\n", pocket.get("pocket_id", pocket["pdb_id"]), "->", pocket["status"])
    for reason in pocket.get("reasons", []):
        print("  Review:", reason)
    for region in pocket.get("regions", []):
        print(" ", region["region_id"], "atoms:", len(region["elements"]),
              "charge:", region["net_charge"], "ready:", region["ready_for_integrals"])
        for warning in region["warnings"]:
            print("   ", warning)

# Keep retrieval errors separate from scientific exclusions.
print("Retrieval errors:", result["retrieval_errors"])
print("Excluded entries:", result["excluded_entries"])

## Input for the next module

The handoff includes all **chemically prepared** scenarios, including the human regions with structural-density warnings, as requested. It does not clear those warnings or change `ready_for_integrals`. Each input carries its PDB ID, review status and warnings so the next module can track the uncertainty. Chemistry failures remain blocked.

This creates PySCF-compatible input dictionaries only; it does not start a calculation. With the validated inputs, expect nine scenarios: three each from 1JFF, 9WDA and 9WDB, of which six carry review flags.

Downstream order: compute initial orbitals/integrals → select the electronic active space → transform integrals → map operators. Do not rank differently charged protonation scenarios by their raw total energies.


In [ ]:
# Include structural-warning cases explicitly, preserving their review metadata.
next_stage_inputs = [
    {"pdb_id": pocket["pdb_id"],
     "pocket_id": pocket["pocket_id"],
     "region_id": region["region_id"],
     "molecule_kwargs": as_pyscf_kwargs(
         region, basis="def2-svp", allow_structural_warnings=True),
     "orbital_selection_policy": region["orbital_selection_policy"],
     "chemistry_checks_passed": region["chemistry_checks_passed"],
     "ready_for_integrals": region["ready_for_integrals"],
     "requires_structural_review": not region["ready_for_integrals"],
     "warnings": list(region.get("warnings", [])),
     "protonation_scenario": region.get("protonation_scenario", {}),
     "readiness_scope": region.get("readiness_scope")}
    for pocket in result["pockets"]
    for region in pocket.get("regions", [])
    if region.get("chemistry_checks_passed", False)
]
print("Prepared inputs included:", len(next_stage_inputs))
print("Inputs carrying structural-review flags:",
      sum(item["requires_structural_review"] for item in next_stage_inputs))


## References and limitations

- [1JFF](https://www.rcsb.org/structure/1JFF), [9WDA](https://www.rcsb.org/structure/9WDA), [9WDB](https://www.rcsb.org/structure/9WDB)
- [Human tubulin/paclitaxel primary study](https://www.nature.com/articles/s41589-026-02204-2)
- [RCSB Search API](https://search.rcsb.org/), [Gemmi documentation](https://gemmi.readthedocs.io/en/latest/), [RDKit documentation](https://www.rdkit.org/docs/)

The optional search helper has a result cap and reports truncation. No exhaustive database scan was performed. Preparation supports standard amino acids and complete TA1; it does not rebuild missing experimental heavy atoms, perform docking, estimate pKa, add solvent, reconstruct an infinite lattice, or select final molecular orbitals. Errors are returned per site, so one failed site does not discard other results. Current scope is the three approved entries; new structures require scientific review of the same assumptions.
